# 22. The invoice numbers that sort like words

**Track:** sql | **Difficulty:** easy | **Tags:** data types, casting, string comparison, joins

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("022-text-invoice-numbers")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "invoices": """
UEFSMRUEFZIUFd4LTBXgAhUAEgAAiQpQBAAAADAwNDkCAAAANzIDAAAAMTMzARUMMTAyNw0IADEBHQQ2MQUdADEFFQwwMDI4CR0A
MA0IBDEwARAIMDAwBT0IMDA0DRAAMgUQDDEwMjUJIAA0CUcEMzYBDwAxDS8MMTAxNAkQBDIzBWYEMjkJBwAxDRYJDwAzBXwMMTAx
NQUeEWMENDQFoBg1AgAAADk3AQYANw1tADUFZggwMDUFhQAwGdINtQAwDY0NrBXcCRcAMAU+EDEwMDIEJSUEMzIBXAA5BTwMMDAz
NAWFADIpNAQxOAUOADMNeQQzMAUPADEJuAAxDccVHgXICDAwMA0YBDI2AVkAOA1nADAJNQAyBZUQMTAxNgMhlQAwBQ8MMDAxOQGN
CDEwMQ14CQcAMyWjADctRQAwJREANxEqDf0AMAmxEc4ANQ06BdUANwXVADE1AgAzBTwIMDAxCTQAMgkHADMJmgAzEWcANwG3BDkw
AQYANhEUCSMAMEVQADgNegA1DV4JOQAwDfAAOS0sADIFggQ2MwFHADglcwgwMDAlIgQxMDEFADMJZQUWCDAwNQ0IEcsANAWJADgF
FggxMDIFiwQwMCl2CDAwNSUcADgFkgA5BQYANw2YADQNmAAxLfAFRwQ3NylwBRwEMDAt2QAxBYEAOAVLADkpOgAxDcAANAkPADAF
DwgxMDEFOQAxLakgMDAzNQIAAAA4SbMAMQXmADkFBgA4MdIt+AAyBZwAOSUCCDEwMg0WADUpewAwaRgAMw1dUQ4p+AA4DfIANA2D
SbkIMDA0TTsFtgA3SZMAMgVgCDAwNg1+ADMJWWkSDDAwMzYBrgA2DSUANC0pEVEYNTUCAAAANgWFADB181FhADIlJAA2Jc8AOQ2T
ADEFSgQxMCkeCDAwMCkIiVsIMDAyRaoANkk1ADEFDQA3DTIAMyUlADltIwAzKWUAMg2RADUlLAA4aXVN7wUqBDAwMZcANQ3wJRQI
MDAzCSwRkAAzDS4kMjUEAAAAMDAwOBUAFfICFfoCLBXgAhUQFQYVBhw2ACgCOTkYBDAwMDEREQAAALkB8LgDAAAA4AIBCC0AAQID
BAUGBwgJCgsMDQ4PEBESExQVFhcYGRobHB0eHyAhIiMkJSYnKCkqKywtLi8wMTIzNDU2Nzg5Ojs8PT4/QEFCQ0RFRkdISUpLTE1O
T1BRUlNUVVZXWFlaW1xdXl9gYWJjZGVmZ2hpamtsbW5vcHFyc3R1dnd4eXp7fH1+f4CBgoOEhYaHiImKi4yNjo+QkZKTlJWWl5iZ
mpucnZ6foKGio6SlpqeoqaqrrK2urxUEFZADFeIBTBUyFQASAADIAQQLAAkBAAUJBwQAEQ0IAA4NCAAVDQgACA0IAA0NCAAHDQgA
Eg0IABYNCAAYDQgACQ0IABcNCAAEDQgAAg0IABMNCAAKDQgABg0IAAwNCAABDQgAGQ0IABANCAAPDQg8AwAAAAAAAAAUAAAAAAAA
ABUAFe4BFfQBLBXgAhUQFQYVBhwYCBkAAAAAAAAAGAgBAAAAAAAAABYAKAgZAAAAAAAAABgIAQAAAAAAAAAREQAAAHfwdgMAAADg
AgEFLSCIQUoxZ6CkwgJksOYedFEg1BAqA4EjCSzznSHPQCaRcOgKtFRkbDJstrsMWnKeVJaoDKkCQnJR2TdYkQWwkmgRZ8aVLgOt
VRGFgNOeV0gKhhLXEklG3rVanWSN2Ra7KGJUnJx4ISkeKFYuMyApFQQV4A0V6g1MFeACFQASAADwBvRvAwAAC/L8AAAcncgAADab
TAAANwXwAAA05DwAAAF7TAAAEne8AAAV3BgAAAViIAAAOlRsAAAq9HwAADZ1aAAAL4AoAAAnEMgAACLHhAAADTmMAAAs8LAAADSO
sAAAG+8sAAAjVIgAABcBdAAAArdQAAApmgAAAAzl9AAAHtucAAAzh8wAADndrAAANafwAAA20fwAABTgiAAADZoIAAAgTrQAACrG
yAAAHyY4AAAz54AAAAVYXAAAG3WwAAAcbiAAADrClAAANqj4AAA4OPgAABaxxAAALsroAAAh+zgAACHIDAAAKFNwAAAZlAwAACWu
4AAAKTG0AAAHHXwAABxD8AAABy90AAA6rkQAADYswAAAC5KAAAA14DAAADvzqAAAJsYsAAAPhQwAAA2B0AAAE8VUAAA3y5gAADAU
mAAAJ83YAAALa9QAABgHLAAAOBXQAAAHrswAACxeNAAAETnEAAAtBQAAAAHpEAAABRtsAAAom7QAACatLAAAFHcQAAAXOhgAACAu
SAAAMU3gAAAIC2AAAAG6lAAAIB9wAAANhoAAADFTvAAABlr0AAAHpEAAABHgLAAAB9j8AAAHjsQAABFEUAAAGXycAAABWhgAAAp0
kAAAJhqwAAALAyQAADazIAAAObroAAAI0dAAAByMmAAAOX+IAAA1CoQAACgqCAAACE0AAAA8qCAAACcpAAAADdgkAAAw5fgAACzy
QAAAHbOEAAAPqvAAABPApAAACARYAAA4wCAAABCUJAAABde0AAA3jLQAADA64AAABXksAAAzi7QAAATAaAAAOQ/QAAA4VqgAACJs
HAAAIDrIAAAQiZgAACy/3AAAN73sAAAV0xwAADgz5AAAOTNcAAAbQCwAADML+AAAHpsoAAAiR8gAABSnHAAADjDQAAALsJQAACi6
9AAAO5fcAAArmuQAABtPBAAAEMKgAAAwDyAAABsZ5AAAKrbEAAArE1gAACnfJAAAELpsAAAKbxgAACU+xAAAF6fcAAAzWbQAACTt
IAAAHSQoAAAUpYwAAAOFkAAAJrT8AAAKmnQAABm3NAAAGGZ8AAAxlcAAAB7DAAAAN2AsAAAXhXwAABmNaAAACPXAAAAbKSAAADZk
OAAAJp2MAAAfg5QAABysoAAAL5FYAAAQN5AAACqp4AAAFrkwAAAclMwVABXyAhX6AiwV4AIVEBUGFQYcGAUAADyoIBgFAAABWhgW
ACgFAAA8qCAYBQAAAVoYEREAAAC5AfC4AwAAAOACAQgtAAECAwQFBgcICQoLDA0ODxAREhMUFRYXGBkaGxwdHh8gISIjJCUmJygp
KissLS4vMDEyMzQ1Njc4OTo7PD0+P0BBQkNERUZHSElKS0xNTk9QUVJTVFVWV1hZWltcXV5fYGFiY2RlZmdoaWprbG1ub3BxcnN0
dXZ3eHl6e3x9fn+AgYKDhIWGh4iJiouMjY6PkJGSk5SVlpeYmZqbnJ2en6ChoqOkpaanqKmqq6ytrq8VBBlMNQAYBnNjaGVtYRUG
ABUMJQIYCmludm9pY2Vfbm8lAEwcAAAAFQQlAhgLY3VzdG9tZXJfaWQAFQ4VChUCGAZhbW91bnQlChUEFRQsXBUEFRQAAAAW4AIZ
HBk8JgAcFQwZNQAGEBkYCmludm9pY2Vfbm8VAhbgAhbuFxbCDyaIDCYIHDYAKAI5ORgEMDAwMRERABksFQQVABUCABUAFRAVAgA8
FpIJGQYZJgDgAgAAACYAHBUEGTUABhAZGAtjdXN0b21lcl9pZBUCFuACFqIGFvoEJswRJsoPHBgIGQAAAAAAAAAYCAEAAAAAAAAA
FgAoCBkAAAAAAAAAGAgBAAAAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgDgAgAAACYAHBUOGTUABhAZGAZhbW91bnQVAhbg
AhbgERbyESbQIibEFBwYBQAAPKggGAUAAAFaGBYAKAUAADyoIBgFAAABWhgREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYA4AIAAAAW
8C8W4AImCBauJgAZHBgMQVJST1c6c2NoZW1hGMwCLy8vLy8vQUFBQUFRQUFBQUFBQUtBQXdBQmdBRkFBZ0FDZ0FBQUFBQkJBQU1B
QUFBQ0FBSUFBQUFCQUFJQUFBQUJBQUFBQU1BQUFDUUFBQUFRQUFBQUFRQUFBQ00vLy8vQUFBQkJ4QUFBQUFnQUFBQUJBQUFBQUFB
QUFBR0FBQUFZVzF2ZFc1MEFBQUlBQXdBQkFBSUFBZ0FBQUFLQUFBQUFnQUFBTVQvLy84QUFBRUNFQUFBQUNRQUFBQUVBQUFBQUFB
QUFBc0FBQUJqZFhOMGIyMWxjbDlwWkFBSUFBd0FDQUFIQUFnQUFBQUFBQUFCUUFBQUFCQUFGQUFJQUFZQUJ3QU1BQUFBRUFBUUFB
QUFBQUFCQlJBQUFBQWdBQUFBQkFBQUFBQUFBQUFLQUFBQWFXNTJiMmxqWlY5dWJ3QUFCQUFFQUFRQUFBQT0AGCBwYXJxdWV0LWNw
cC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRk8HAAAHAAAHAAAACADAABQQVIx
""",
    "payments": """
UEFSMRUEFfAOFd4HTBXuARUAEgAAuAcI6gMABQEAhwUGCAAA8Q0QAFQNEAAXDQgAJg0IAHMNCAB8DQgAXQ0IAEsNCACCDQgAVg0I
ABQNCABtDQgA6w1gAPwNCAAxDRgAXg0IAPoNGAANDRAAHA0IAO4NGAB7DRAAaw0IACMNCAA9DQgAIg0IAEUNCAAqDQgAiA0IADwN
CACEDQgATQ0IAAkNCAQ6AAkBAGgJBwQAiQ0IDX8EAB4NEAA3DQgAaQ0IAP0NIQAuDRAAAQ0IAFMNCAAFDQgEAQQJCQASCQcEAIEN
CAB2DQgA/w1IAEcNEAD5DRAAWQ0QAEQNCAD0DRgABA1QAPcNEABQDSAAig0IDR8EAGcNEAD2DSgAQw0QAFgNCAB4DQgA7A0gAFsN
EABRDQgAag0IADMNCABXDQgAcA0IABgNCAA5DQgAEA0IAAsNgQBxDRAAfg0IAB8NCAAGDSAAfw0QBPADCQkADwkHBAD1DRAAVQ0Q
AA4NCAAWDQgA7Q0gADsNEAD+DRAAJQ0QAEgNCABaDQgAYg0IAGANCABADQgAcg0IAAoNkAAVDRAAZQ0IAO8NWACADRAAAw0oAIsN
EADoDSAAbw0QAAsNCAAIDQgAPg0IABkNCABsDQgAjA0IAAINCABCDQgAJA0IAE8NCDw2AAAAAAAAAEEAAAAAAAAAFQAV5AEV6gEs
Fe4BFRAVBhUGHBgICwQAAAAAAAAYCAEAAAAAAAAAFgAoCAsEAAAAAAAAGAgBAAAAAAAAABERAAAAcvBxAwAAAO4BAQcfgIBgQCgY
DoiEYsFoOB6QiGRCqVgumIxmw+l4PqCQaEQqmU6olGrFarlesJhsRqvZbricbsfr+X7AoHBILBqPyKRyyWw6n9CodEqtWq/YrHbL
7Xq/4LB4TC6bz+i0es1uu9/wuHxOr9sBFQQVqAQVsgRMFYoBFQASAACUAvQTAcZOAADITgAAs04AAO5OAAC8TgAA6k4AAKNOAACF
TgAA5E4AAIdOAADPTgAAjU4AALFOAADvTgAAmk4AALJOAACmTgAA7U4AAIROAACVTgAAyk4AAOdOAACCTgAAik4AAIBOAADVTgAA
v04AAHlOAACUTgAAfE4AALlOAACiTgAAfU4AALdOAACMTgAAmE4AANROAADRTgAAg04AANZOAADdTgAAx04AAKFOAAC0TgAAzE4A
AKpOAADNTgAAiU4AAK5OAACpTgAA3k4AAJ5OAAB+TgAAm04AAMJOAADsTgAA3E4AAL1OAAC6TgAAl04AAJ9OAADOTgAA0k4AAOBO
AADDTgAAgU4AANBOAACsTgAAjk4AABUAFeQBFeoBLBXuARUQFQYVBhwYBO9OAAAYBHlOAAAWACgE704AABgEeU4AABERAAAAcvBx
AwAAAO4BAQcfgIBgMCAUDAdEQrFIMBANxwMSkUwoFcsFI8hoNpxOwPMBFTiSkGiEIZVMJ5ThkVKtWC2B6+UhrVywmEwwo5lGtBrD
lrkpLpDEyIQznXK6iaW12vEYnF7H5/iBVgoXUPQLCjVDxU/mINoAFQQZPDUAGAZzY2hlbWEVBAAVBCUCGAppbnZvaWNlX25vABUC
JQIYB3BhaWRfb24lDExsAAAAFu4BGRwZLCYAHBUEGTUABhAZGAppbnZvaWNlX25vFQIW7gEW+hEW7gomiAgmCBwYCAsEAAAAAAAA
GAgBAAAAAAAAABYAKAgLBAAAAAAAABgIAQAAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYA7gEAAAAmABwVAhk1AAYQGRgH
cGFpZF9vbhUCFu4BFpIHFqIHJsoPJvYKHBgE704AABgEeU4AABYAKATvTgAAGAR5TgAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkm
AO4BAAAAFowZFu4BJggWkBIAGRwYDEFSUk9XOnNjaGVtYRiAAi8vLy8vN2dBQUFBUUFBQUFBQUFLQUF3QUJnQUZBQWdBQ2dBQUFB
QUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFJQUFBQk1BQUFBQkFBQUFNei8vLzhBQUFFSUVBQUFBQ0FBQUFBRUFBQUFB
QUFBQUFjQUFBQndZV2xrWDI5dUFBQUFCZ0FJQUFZQUJnQUFBQUFBQUFBUUFCUUFDQUFHQUFjQURBQUFBQkFBRUFBQUFBQUFBUUlR
QUFBQUpBQUFBQVFBQUFBQUFBQUFDZ0FBQUdsdWRtOXBZMlZmYm04QUFBZ0FEQUFJQUFjQUNBQUFBQUFBQUFGQUFBQUEAGCBwYXJx
dWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRksHAAAHAAAAFYCAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "022-text-invoice-numbers"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

invoices = spark.read.parquet((_data_dir / "invoices.parquet").resolve().as_posix())
invoices.createOrReplaceTempView("invoices")
payments = spark.read.parquet((_data_dir / "payments.parquet").resolve().as_posix())
payments.createOrReplaceTempView("payments")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'customer_id': 'a31c620fe04134816ddf749d4168c191152bd2ff9eec30f957859137d7dcd2c5',
                         'invoice_count': 'da58999db4c4a3d4aa89f302c4298b0eb4c472751f96c132a0254dec16cae0d2',
                         'latest_invoice_no': '30eff0fc2023b1918d07616d92ab27ef85eefd2081c6edef62ec57f4539d59a0',
                         'paid_invoices': '12b6a787dc400c83a28d9b0d6d75c328ceffd0cc67f0167ab743b250fd97e118'},
 'columns': [['customer_id', 'bigint'],
             ['invoice_count', 'bigint'],
             ['latest_invoice_no', 'bigint'],
             ['paid_invoices', 'bigint']],
 'fingerprint': 'f4e837403d85b4642336cc49e307f810a71b667b0d8812a571afa565d8b5ac21',
 'order_matters': False,
 'row_count': 25}

check = make_check(EXPECTED)

## The situation

Invoice numbers are plain running numbers: a higher number means a later invoice. In
the `invoices` table they are stored as text, because the old billing system printed
them with leading zeros (`0042`). The `payments` table, built later, stores the same
numbers as integers.

A customer statement shows the customer's latest invoice. One customer complained
that their statement stops at invoice 99, although they have received invoice 1000.

## Input tables

### `invoices`

One row per invoice.

| Column | Type | Description |
|---|---|---|
| `invoice_no` | STRING | The invoice number as text. Older invoices are padded with zeros to four digits (`0042`); newer ones are not (`105`, `1003`). |
| `customer_id` | BIGINT | The customer who was invoiced. |
| `amount` | DECIMAL(10,2) | Invoice value in rupees. |

### `payments`

One row per paid invoice. An invoice is paid at most once.

| Column | Type | Description |
|---|---|---|
| `invoice_no` | BIGINT | The number of the invoice that was paid. |
| `paid_on` | DATE | Date of the payment. |

## Requirement

Return one row per customer.

- `invoice_count` is the number of invoices of the customer.
- `latest_invoice_no` is the customer's highest invoice number, compared as a
  number. `1000` is later than `99`, and `0042` is invoice 42.
- `paid_invoices` is how many of the customer's invoices have a payment. Invoice
  `0042` is paid if `payments` has a row for 42.

## Expected output

One row per customer.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | |
| `invoice_count` | BIGINT | |
| `latest_invoice_no` | BIGINT | The highest invoice number, as a number. |
| `paid_invoices` | BIGINT | 0 if none are paid. |

In [ ]:
invoices.show(5)
payments.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `invoices`, `payments`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)